In [1]:
from lsst.daf.butler import Butler
from useful_functions_global import *
import pandas as pd
import numpy as np
import healpy as hp
from astropy.table import Table, vstack
import fitsio
import matplotlib.pyplot as plt
from ugali.utils.projector import match
#from astropy.coordinates import SkyCoord
#import astropy.units as u
import gc
#from concurrent.futures import ThreadPoolExecutor
import time


my_path =  '/sdf/data/rubin/user/kexcel/'
my_plotspath = my_path + 'plots/'

survey = 'dp2'
## unembargoed repo
mainrepo="/repo/main"

## After the migration, w_2026_19 (or later) or v30_0_8_rc4 will be needed to use dp2_prep <- from Slack
if survey == 'dp2':
    repo = "dp2_prep"
    collection = ['LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1',
                  'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2',
                  'LSSTCam/runs/DRP/DP2/v30_0_6_rc1/DM-53881/stage3',
                  'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4']
    #field = 'edfs'
    #collection = [f'u/dtaranu/DM-50135/DP2/v30_0_8/matched_{field}']
    skymap = 'lsst_cells_v2'
elif survey == 'dp1':
    repo = "dp1"
    collection = ["LSSTComCam/DP1"]
    skymap = 'lsst_cells_v1'

In [2]:
def load_lsst_data(field_or_tract):
    print('Start LSST data load')
    start_time = time.perf_counter()

    INCOLS = [
        'coord_ra',
        'coord_dec',
        'detect_isIsolated',
        'refExtendedness',
        'tract', 'patch'
    ]
    if survey=='dp1':
        bands = 'ugrizy'
        for band in bands:
            INCOLS += [
                f'{band}_psfFlux',
                f'{band}_cModelFlux',
                f'{band}_cModelFluxErr',
                f'{band}_psfFluxErr',
                f'{band}_extendedness',
                f'{band}_extendedness_flag',
                f'{band}_psfFlux_flag'
            ]
            INCOLS += [f'{band}_sizeExtendedness', f'{band}_sizeExtendedness_flag']
    ## new classifier from Dan Taranu = {band}_model_extendedness
    ## old classifier from DM is {band}_extendedness or {band}_SizeExtendedness - these might become depreciated in DP2 or DR1
    elif survey=='dp2':
        INCOLS += ['griz_model_extendedness']
        bands = 'grizy'
        for band in bands:
            INCOLS += [
                f'{band}_psfFlux',
                f'{band}_cModelFlux',
                f'{band}_cModelFluxErr',
                f'{band}_psfFluxErr',
                f'{band}_extendedness',
                f'{band}_extendedness_flag',
                f'{band}_psfFlux_flag'
            ]
            INCOLS += [f'{band}_model_extendedness']
    
    butler = Butler(repo, collections=collection)
    registry=butler.registry
    
    ## this is for loading data in by field name
    if type(field_or_tract) == str:
        tract_list = get_tract(field_or_tract)
        raw_lsst_list = []
        for i in range(len(tract_list)):
            tract = tract_list[i]
            raw_lsst_list.append(butler.get('object', dataId={'skymap': skymap, 'tract': tract}, collections=collection,
                                parameters={"columns":INCOLS}))
        raw_lsst = vstack(raw_lsst_list)
        del raw_lsst_list,tract_list
        gc.collect()
    ## for loading data in by tract number
    elif type(field_or_tract) == int:
        raw_lsst = butler.get('object', dataId={'skymap': skymap, 'tract': field_or_tract}, collections=collection,
                                parameters={"columns":INCOLS})
    # End the timer
    end_time = time.perf_counter()
    # Calculate and print elapsed time
    print(f'End LSST data load. Execution time: {(end_time - start_time):.6f} seconds')
    return raw_lsst

In [3]:
def load_euclid_file(pix):
    columns_to_load = ['RIGHT_ASCENSION','DECLINATION',
                       'POINT_LIKE_PROB', 'POINT_LIKE_FLAG',
                       'ELLIPTICITY', 'MUMAX_MINUS_MAG',
                       'FLUX_VIS_PSF', 'FLUXERR_VIS_PSF',
                       'SPURIOUS_FLAG','DET_QUALITY_FLAG', 'FWHM']
    num = 2
    for band in ['VIS', 'Y', 'J', 'H']:
        columns_to_load += [f'FLAG_{band}',f'FLUX_{band}_{num}FWHM_APER', f'FLUXERR_{band}_{num}FWHM_APER']
    try:
        euclid_path = "/sdf/group/rubin/shared/euclid/q1/catalogs/"
        raw_euclid = Table(fitsio.read(euclid_path + f"euclid_q1_mer_final_{pix:05d}.fits", columns = columns_to_load))
    except:
        print('Exception')
        raw_euclid = Table()
    return raw_euclid

In [4]:
def euclid_merge(lsst_datafile, field_or_tract, visualization = False):
    ## coarser resolution to load in Euclid files, which follow nside = 64 pixel name convention
    nside = 64
    lsst_ra = lsst_datafile['coord_ra']
    lsst_dec = lsst_datafile['coord_dec']
    ## converting our queried LSST ra and dec into pixels
    lsst_pix64 = hp.ang2pix(nside, lsst_ra, lsst_dec, lonlat=True, nest=True)
    lsst_upix64 = np.unique(lsst_pix64)
    
    print('Start Euclid data load')
    start_time = time.perf_counter()
    #with ThreadPoolExecutor() as executor:
    #    euclid = vstack(list(executor.map(load_euclid_file, lsst_upix64)))
    euclid_list = []
    for pix in lsst_upix64:
        euclid_list.append(load_euclid_file(pix))
    euclid = vstack(euclid_list)
    if (len(euclid['RIGHT_ASCENSION']) == 0) or (len(euclid['DECLINATION']) == 0):
        print('No euclid data for ', field_or_tract)
        return
    del nside, lsst_pix64, lsst_upix64, euclid_list
    gc.collect()
    end_time = time.perf_counter()
    print(f'End Euclid data load. Execution time: {(end_time - start_time):.6f} seconds')

    print('Start area overlap masking')
    start_time = time.perf_counter()
    ## now get greater resolution to match up
    NSIDE=4096
    ## get the pixels of LSST data
    lsst_upix4096, lsst_cts = np.unique(hp.ang2pix(NSIDE, lsst_ra, lsst_dec, lonlat=True), return_counts=True)
    ## then get the pixels of Euclid data
    euclid_pix4096 = hp.ang2pix(NSIDE, euclid['RIGHT_ASCENSION'], euclid['DECLINATION'], lonlat=True)
    ## Euclid has more coverage right now. We only keep the sources that lie in the LSST coverage
    mask = np.isin(euclid_pix4096, lsst_upix4096) #[lsst_cts > 8])
    euclid_field = euclid[mask]
    del NSIDE, mask, lsst_upix4096, lsst_cts, euclid_pix4096, euclid
    gc.collect()
    end_time = time.perf_counter()
    print(f'End area overlap masking. Execution time: {(end_time - start_time):.6f} seconds')
    
    print('Start catalog matching')
    start_time = time.perf_counter()
    ## match() is from ugali tools -- matching LSST and Euclid sources
    if len(euclid_field['RIGHT_ASCENSION']) == 0:
        return
    indexlsst, indexeuclid, ds = match(lsst_ra, lsst_dec, 
                                       euclid_field['RIGHT_ASCENSION'], euclid_field['DECLINATION'], 
                                       tol = 0.0003)
    #print('index lsst:', '\n', indexlsst[0:20])
    #print('index euclid:', '\n', indexeuclid[0:20])
    matches_lsst = lsst_datafile[indexlsst]
    unmatched_lsst = lsst_datafile[~indexlsst]
    #print(matches_lsst.columns)
    matches_euclid = euclid_field[indexeuclid]
    unmatched_euclid = euclid_field[~indexeuclid]
    if len(matches_lsst) != len(matches_euclid):
        print("Something isn't right: those lengths don't match")
    del indexlsst, indexeuclid, lsst_ra, lsst_dec
    gc.collect()
    end_time = time.perf_counter()
    print(f'End catalog matching. Execution time: {(end_time - start_time):.6f} seconds')

    if visualization == True:
        ## Match Verification
        b = 60
        #1D histogram of matches and not matches
        fig, ax = plt.subplots(1,1, figsize=(13,5))
        match_vis_mag = flux2mag(matches_euclid['FLUX_VIS_2FWHM_APER']*(10**3))
        unmatch_vis_mag = flux2mag(unmatched_euclid['FLUX_VIS_2FWHM_APER']*(10**3))
        total_vis_mag = flux2mag(euclid_field['FLUX_VIS_2FWHM_APER']*(10**3))
        plt.hist(match_vis_mag, bins = b, histtype = 'step', color='b', label = 'Matched Euclid Sources')
        plt.hist(unmatch_vis_mag, bins = b, histtype = 'step', color='r', label = 'Unmatched Euclid Sources')
        plt.hist(total_vis_mag, bins = b, histtype = 'step', color='k', label = 'Total Euclid Sources')
        plt.xlabel('FLUX_VIS_2FWHM_APER mag')
        plt.xlim(16,36)
        plt.ylabel('Number counts')
        plt.yscale('log')
        plt.title(f'Tract {tract}: Euclid Source Match/Unmatch')
        plt.legend()
        plt.show()
        
        match_i_mag = flux2mag(matches_lsst['i_psfFlux'])
        unmatch_i_mag = flux2mag(unmatched_lsst['i_psfFlux'])
        total_i_mag = flux2mag(lsst_datafile['i_psfFlux'])
        fig, ax = plt.subplots(1,1, figsize=(13,5))
        plt.hist(match_i_mag, bins = b, histtype = 'step', color='c', label = 'Matched LSST Sources')
        plt.hist(unmatch_i_mag, bins = b, histtype = 'step', color='r', label = 'Unmatched LSST Sources')
        plt.hist(total_i_mag, bins = b, histtype = 'step', color='k', label = 'Total LSST Sources')
        plt.xlabel('i_psfFlux mag')
        plt.xlim(16,36)
        plt.ylabel('Number counts')
        plt.yscale('log')
        plt.title(f'Tract {tract}: LSST Source Match/Unmatch')
        plt.legend()
        plt.show()

        #2D histogram of matches in Euclid and LSST, does it look the same?
        fig, ax = plt.subplots(1,2, figsize=(13,5))
        _, _, _, im = ax[0].hist2d(matches_euclid['RIGHT_ASCENSION'], matches_euclid['DECLINATION'], bins=100)
        plt.colorbar(im, ax=ax[0])
        ax[0].set(title = f'Tract {tract}: Matches in Euclid', ylabel = "Dec (deg)", xlabel = "RA (deg)")
        ax[0].invert_xaxis()
        _, _, _, im = ax[1].hist2d(matches_lsst['coord_ra'], matches_lsst['coord_dec'], bins=100)
        plt.colorbar(im, ax=ax[1])
        ax[1].set(title = f'Tract {tract}: Matches in LSST', ylabel = "Dec (deg)", xlabel = "RA (deg)")
        ax[1].invert_xaxis()
        plt.show()

        '''
        #histogram of separation
        ds = ds * 3600 #ds is in degrees, want to plot in arcsecs
        #I forced in the function that matches would be <1"
        plt.hist(ds, histtype='step', range=(0,1))
        plt.xlabel('separation [arcsec]')
        plt.title(f'Tract {tract}: Matched Source Separation')
        plt.tight_layout()
        plt.show()

        #checking that dec and DECLINATION relation is slope of 1
        plt.scatter(merged_df['coord_dec'],merged_df['DECLINATION'],)
        '''

    ## now merging our matches into one catalog with all LSST and Euclid columns
    print('Start catalog merge')
    start_time = time.perf_counter()
    lsst_df = matches_lsst.to_pandas()
    euclid_df = matches_euclid.to_pandas()
    lsst_df['_match_id'] = np.arange(len(lsst_df))
    euclid_df['_match_id'] = np.arange(len(euclid_df))
    merged_df = pd.merge(lsst_df, euclid_df, on='_match_id').drop(columns='_match_id')
    del lsst_df, euclid_df, matches_lsst, matches_euclid, unmatched_lsst, unmatched_euclid, ds
    gc.collect()
    merged_df.to_parquet(my_path + f'/euclid_data/q1/{field_or_tract}_{survey}_euclid_merged.parquet', 
                         index=False, compression = 'snappy')
    end_time = time.perf_counter()
    print(f'End catalog merge. Merged catalog saved. Execution time: {(end_time - start_time):.6f} seconds')

    ## comment/uncomment as needed for debugging
    del merged_df, euclid_field, lsst_datafile
    gc.collect()
    #return merged_df #, euclid_field, lsst_datafile

In [5]:
tract_list = get_tract('EDFS')
for i in range(len(tract_list)):
    tract = tract_list[i]
    print(tract)
    euclid_merge(load_lsst_data(tract), tract, 
                             visualization = False)
    print(' ')

2078
Start LSST data load
End LSST data load. Execution time: 1.716527 seconds
Start Euclid data load
Exception
Exception
Exception
Exception
End Euclid data load. Execution time: 46.628397 seconds
Start area overlap masking
End area overlap masking. Execution time: 0.166954 seconds
Start catalog matching
End catalog matching. Execution time: 2.333962 seconds
Start catalog merge
End catalog merge. Merged catalog saved. Execution time: 0.220218 seconds
 
2079
Start LSST data load
End LSST data load. Execution time: 1.341420 seconds
Start Euclid data load
Exception
Exception
End Euclid data load. Execution time: 145.435947 seconds
Start area overlap masking
End area overlap masking. Execution time: 0.286835 seconds
Start catalog matching
End catalog matching. Execution time: 4.546411 seconds
Start catalog merge
End catalog merge. Merged catalog saved. Execution time: 1.000874 seconds
 
2080
Start LSST data load
End LSST data load. Execution time: 1.687851 seconds
Start Euclid data load
E

In [6]:
tract_list = get_tract('ECDFS')
for i in range(len(tract_list)):
    tract = tract_list[i]
    print(tract)
    euclid_merge(load_lsst_data(tract), tract, visualization = False)
    print(' ')

4848
Start LSST data load
End LSST data load. Execution time: 1.065034 seconds
Start Euclid data load
End Euclid data load. Execution time: 63.053268 seconds
Start area overlap masking
End area overlap masking. Execution time: 0.275472 seconds
Start catalog matching
End catalog matching. Execution time: 1.446537 seconds
Start catalog merge
End catalog merge. Merged catalog saved. Execution time: 2.427375 seconds
 
4849
Start LSST data load
End LSST data load. Execution time: 1.084213 seconds
Start Euclid data load
End Euclid data load. Execution time: 49.608739 seconds
Start area overlap masking
End area overlap masking. Execution time: 0.284467 seconds
Start catalog matching
End catalog matching. Execution time: 1.371606 seconds
Start catalog merge
End catalog merge. Merged catalog saved. Execution time: 2.788575 seconds
 
5063
Start LSST data load
End LSST data load. Execution time: 2.897486 seconds
Start Euclid data load
End Euclid data load. Execution time: 43.459089 seconds
Start 

In [ ]:
## CODE FOR IF THE MERGE IS WEIRD

i_mag = flux2mag(lsst_datafile['i_psfFlux'])
vis_mag = flux2mag(euclid_field['FLUX_VIS_PSF']*10**3)

print(len(i_mag))

lsst_datafile1 = lsst_datafile #[(i_mag < 22)]
euclid_field1 = euclid_field #[(vis_mag < 22)]

plt.scatter(euclid_field1['RIGHT_ASCENSION'], euclid_field1['DECLINATION'], 
            marker = '+', label = 'Euclid Sources', #c = flux2mag(euclid_field1['FLUX_VIS_PSF']*10**3), 
           )
plt.scatter(lsst_datafile1['coord_ra'], lsst_datafile1['coord_dec'], 
            marker = 'x', label = 'LSST Sources', #c = flux2mag(lsst_datafile1['i_psfFlux']),
           )
plt.xlim(59.85, 59.84)
plt.xlabel('RA (deg)')
plt.ylim(-48.55,-48.54)
plt.ylabel('DEC (deg)')
#plt.colorbar()
plt.legend()
plt.title('Euclid and LSST before matching, \n restricted i_mag & vis_mag < 22')
plt.show()

ra_diff = (merged_df['RIGHT_ASCENSION'] - merged_df['coord_ra'])*3600
dec_diff = (merged_df['DECLINATION'] - merged_df['coord_dec'])*3600
plt.hist(ra_diff, bins = 100)
plt.title('Merged Catalog RA difference')
plt.xlabel('Euclid RA - LSST RA (arcsec)')
plt.xlim(-1,1)
plt.show()

plt.hist(dec_diff, bins = 100)
plt.xlim(-0.5,0.5)
plt.xlabel('Euclid DEC - LSST DEC (arcsec)')
plt.title('Merged Catalog DEC difference')
plt.show()